In [1]:
# Imports and Configuration

import os
import re
import json
import time
import math
import logging
import platform
import psutil
import requests
import wikipediaapi
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any
from pathlib import Path
from collections import deque
from dotenv import load_dotenv
from groq import Groq
from ddgs import DDGS
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

MAX_RETRIES = 3
RETRY_DELAY = 2
DEFAULT_MODEL = "llama-3.3-70b-versatile"
DEFAULT_MAX_TOKENS = 2048
DEFAULT_TEMPERATURE = 0.3
SHORT_TERM_LIMIT = 10
MEMORY_RESULTS = 3
MAX_RESULTS_PER_QUERY = 4
MAX_BODY_LENGTH = 300
MAX_TOOL_OUTPUT = 500
CHROMA_PATH = "C:/educational files/advanced_agent/memory/chroma_store"
HISTORY_PATH = "C:/educational files/advanced_agent/memory/search_history.json"
SESSION_LOG_PATH = "C:/educational files/advanced_agent/memory/session_log.json"
EXPORT_PATH = "C:/educational files/advanced_agent/memory/session_export.txt"

AGENT_CAPABILITIES = [
    "Multi-turn conversation with persistent memory",
    "Short-term buffer + ChromaDB long-term vector memory",
    "Semantic memory recall across sessions",
    "10 tools: calculator, wikipedia, web search, file reader, datetime, unit converter, dictionary, weather, csv analyzer, system info",
    "Multi-query web search pipeline with source ranking",
    "Follow-up detection with context injection",
    "Intent classification: chat, tool, search, memory recall, file analysis",
    "Session logging to JSON",
    "Conversation export to .txt",
    "Graceful error recovery with pipeline fallback",
    "Performance tracking per intent type",
]

In [2]:
# Client Initialization

env_path = Path("C:/educational files/advanced_agent/.env")
load_dotenv(dotenv_path=env_path)

def init_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY not found in .env")
    log.info("Groq client initialized successfully")
    return Groq(api_key=api_key)

client = init_client()

2026-06-03 10:38:12,919 [INFO] Groq client initialized successfully


In [3]:
# Master Agent Configuration

@dataclass
class AgentConfig:
    model: str = DEFAULT_MODEL
    max_tokens: int = DEFAULT_MAX_TOKENS
    temperature: float = DEFAULT_TEMPERATURE
    system_prompt: str = (
        "You are a fully autonomous AI agent with memory, tools, and web search capabilities. "
        "You recall past interactions, execute tools, search the web, and reason across multiple steps. "
        "Always use the most appropriate capability for each request. "
        "Be concise, precise, and professional."
    )
    session_token_count: int = field(default=0, repr=False)
    tool_call_count: int = field(default=0, repr=False)
    search_count: int = field(default=0, repr=False)
    memory_hits: int = field(default=0, repr=False)
    total_turns: int = field(default=0, repr=False)
    intent_counts: Dict[str, int] = field(default_factory=lambda: {
        "chat": 0, "tool": 0, "search": 0, "memory_recall": 0, "file_analysis": 0
    }, repr=False)
    response_times: Dict[str, List[float]] = field(default_factory=lambda: {
        "chat": [], "tool": [], "search": [], "memory_recall": [], "file_analysis": []
    }, repr=False)

config = AgentConfig()
log.info(f"Master agent configured — model: {config.model}")

2026-06-03 10:38:13,840 [INFO] Master agent configured — model: llama-3.3-70b-versatile


In [4]:
# Memory System

class ShortTermBuffer:
    def __init__(self, limit: int = SHORT_TERM_LIMIT):
        self.buffer: deque = deque(maxlen=limit)
        self.limit = limit

    def add(self, role: str, content: str) -> None:
        self.buffer.append({"role": role, "content": content})

    def get(self) -> List[Dict[str, str]]:
        return list(self.buffer)

    def clear(self) -> None:
        self.buffer.clear()

    def summary(self) -> str:
        return f"Short term: {len(self.buffer)}/{self.limit} messages"


class LongTermMemory:
    def __init__(self, path: str = CHROMA_PATH):
        Path(path).mkdir(parents=True, exist_ok=True)
        self.client = chromadb.PersistentClient(path=path)
        self.ef = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.get_or_create_collection(
            name="agent_memory", embedding_function=self.ef
        )
        log.info(f"Long term memory ready — stored entries: {self.collection.count()}")

    def store(self, memory_id: str, text: str, metadata: Dict) -> None:
        self.collection.upsert(ids=[memory_id], documents=[text], metadatas=[metadata])

    def query(self, query_text: str, n_results: int = MEMORY_RESULTS) -> List[str]:
        count = self.collection.count()
        if count == 0:
            return []
        results = self.collection.query(query_texts=[query_text], n_results=min(n_results, count))
        return results["documents"][0] if results["documents"] else []

    def clear(self) -> None:
        self.client.delete_collection("agent_memory")
        self.collection = self.client.get_or_create_collection(
            name="agent_memory", embedding_function=self.ef
        )

    def count(self) -> int:
        return self.collection.count()


class MemoryManager:
    def __init__(self, short_term: ShortTermBuffer, long_term: LongTermMemory):
        self.short_term = short_term
        self.long_term = long_term
        self._turn_counter = 0

    def add_exchange(self, user_input: str, reply: str) -> None:
        self.short_term.add("user", user_input)
        self.short_term.add("assistant", reply)
        self._turn_counter += 1
        self.long_term.store(
            memory_id=f"turn_{self._turn_counter}_{int(time.time())}",
            text=f"User: {user_input}\nAssistant: {reply}",
            metadata={"turn": self._turn_counter, "timestamp": int(time.time())}
        )

    def retrieve_context(self, query: str) -> str:
        results = self.long_term.query(query)
        if not results:
            return ""
        return "Relevant memory context:\n" + "\n---\n".join(results)

    def clear_all(self) -> None:
        self.short_term.clear()
        self.long_term.clear()
        self._turn_counter = 0
        log.info("All memory cleared")

    def summary(self) -> str:
        return f"{self.short_term.summary()} | Long term: {self.long_term.count()} entries | Turns: {self._turn_counter}"


short_term = ShortTermBuffer()
long_term = LongTermMemory()
memory = MemoryManager(short_term, long_term)
log.info("Memory system ready")

2026-06-03 10:38:14,189 [INFO] Long term memory ready — stored entries: 3
2026-06-03 10:38:14,192 [INFO] Memory system ready


In [5]:
# Tool Definitions

def calculator(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return f"Result: {result}"
    except Exception as e:
        return f"Calculator error: {e}"

def wikipedia(query: str) -> str:
    try:
        wiki = wikipediaapi.Wikipedia(language="en", user_agent="AutonomousAgent/1.0")
        page = wiki.page(query)
        if not page.exists():
            return f"No Wikipedia page found for: {query}"
        return page.summary[:MAX_TOOL_OUTPUT]
    except Exception as e:
        return f"Wikipedia error: {e}"

def web_search_tool(query: str) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        return "\n".join([f"{r['title']}: {r['body'][:150]}" for r in results])
    except Exception as e:
        return f"Web search error: {e}"

def file_reader(filepath: str) -> str:
    try:
        path = Path(filepath)
        if not path.exists():
            return f"File not found: {filepath}"
        if path.suffix == ".csv":
            df = pd.read_csv(path)
            return f"CSV loaded — shape: {df.shape}\nColumns: {list(df.columns)}\nPreview:\n{df.head(3).to_string()}"
        return path.read_text(encoding="utf-8")[:MAX_TOOL_OUTPUT]
    except Exception as e:
        return f"File reader error: {e}"

def datetime_tool(query: str) -> str:
    now = datetime.now()
    if "time" in query.lower():
        return f"Current time: {now.strftime('%H:%M:%S')}"
    if "day" in query.lower():
        return f"Today is: {now.strftime('%A')}"
    return f"Current datetime: {now.strftime('%Y-%m-%d %H:%M:%S')}"

def unit_converter(query: str) -> str:
    try:
        parts = query.lower().split()
        value, unit_from, unit_to = float(parts[0]), parts[1], parts[3]
        conversions = {
            ("kg","lbs"): lambda x: x*2.20462, ("lbs","kg"): lambda x: x/2.20462,
            ("km","miles"): lambda x: x*0.621371, ("miles","km"): lambda x: x/0.621371,
            ("celsius","fahrenheit"): lambda x: x*9/5+32, ("fahrenheit","celsius"): lambda x: (x-32)*5/9,
            ("meters","feet"): lambda x: x*3.28084, ("feet","meters"): lambda x: x/3.28084,
        }
        key = (unit_from, unit_to)
        if key not in conversions:
            return f"Conversion {unit_from} to {unit_to} not supported."
        return f"{value} {unit_from} = {round(conversions[key](value), 4)} {unit_to}"
    except Exception as e:
        return f"Unit converter error: {e}"

def dictionary(word: str) -> str:
    try:
        r = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}", timeout=5)
        data = r.json()
        if isinstance(data, list):
            m = data[0]["meanings"][0]
            return f"{word} ({m['partOfSpeech']}): {m['definitions'][0]['definition']}"
        return f"No definition found for: {word}"
    except Exception as e:
        return f"Dictionary error: {e}"

def weather(city: str) -> str:
    try:
        url = f"https://wttr.in/{city.replace(' ', '+')}?format=3"
        r = requests.get(url, timeout=5)
        return r.text.strip()
    except Exception as e:
        return f"Weather error: {e}"

def csv_analyzer(filepath: str) -> str:
    try:
        df = pd.read_csv(filepath)
        return f"Shape: {df.shape}\nColumns: {list(df.columns)}\nStats:\n{df.describe().to_string()[:MAX_TOOL_OUTPUT]}"
    except Exception as e:
        return f"CSV analyzer error: {e}"

def system_info(query: str) -> str:
    try:
        cpu = psutil.cpu_percent(interval=1)
        ram = psutil.virtual_memory()
        disk = psutil.disk_usage("/")
        return (
            f"OS: {platform.system()} {platform.release()}\n"
            f"CPU: {cpu}% | RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f}GB ({ram.percent}%)\n"
            f"Disk: {disk.used/1e9:.1f}/{disk.total/1e9:.1f}GB ({disk.percent}%)"
        )
    except Exception as e:
        return f"System info error: {e}"

log.info("10 tools defined successfully")

2026-06-03 10:38:14,241 [INFO] 10 tools defined successfully


In [6]:
# Tool Registry and Executor

TOOL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "calculator":     {"fn": calculator,     "description": "Evaluates math expressions. Input: math expression."},
    "wikipedia":      {"fn": wikipedia,      "description": "Fetches Wikipedia summary. Input: search query."},
    "web_search":     {"fn": web_search_tool,"description": "Searches the web via DuckDuckGo. Input: search query."},
    "file_reader":    {"fn": file_reader,    "description": "Reads txt or csv files. Input: full file path."},
    "datetime_tool":  {"fn": datetime_tool,  "description": "Returns current date, time, or day. Input: query string."},
    "unit_converter": {"fn": unit_converter, "description": "Converts units. Input: '100 km to miles' format."},
    "dictionary":     {"fn": dictionary,     "description": "Returns word definition. Input: single word."},
    "weather":        {"fn": weather,        "description": "Returns current weather. Input: city name."},
    "csv_analyzer":   {"fn": csv_analyzer,   "description": "Analyzes CSV statistics. Input: full file path."},
    "system_info":    {"fn": system_info,    "description": "Returns CPU, RAM, disk usage. Input: any string."},
}

def execute_tool(tool_name: str, tool_input: str) -> str:
    if tool_name not in TOOL_REGISTRY:
        return f"Unknown tool: {tool_name}"
    try:
        log.info(f"Executing tool: {tool_name} | input: {tool_input[:80]}")
        result = TOOL_REGISTRY[tool_name]["fn"](tool_input)
        log.info(f"Tool {tool_name} completed")
        return result
    except Exception as e:
        log.error(f"Tool {tool_name} failed: {e}")
        return f"Tool execution error: {e}"

log.info(f"Tool registry ready — {len(TOOL_REGISTRY)} tools registered")

2026-06-03 10:38:14,269 [INFO] Tool registry ready — 10 tools registered


In [7]:
# Web Search Pipeline

def plan_queries(user_input: str, cfg: AgentConfig) -> List[str]:
    prompt = (
        "Break the following question into 2-3 specific search queries. "
        "Return ONLY a JSON array of strings, nothing else.\n"
        f"Question: {user_input}"
    )
    try:
        response = client.chat.completions.create(
            model=cfg.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
            temperature=0.2
        )
        raw = response.choices[0].message.content.strip()
        cfg.session_token_count += response.usage.total_tokens
        queries = json.loads(raw)
        if isinstance(queries, list) and all(isinstance(q, str) for q in queries):
            log.info(f"Query planner generated {len(queries)} sub-queries")
            return queries
    except Exception as e:
        log.warning(f"Query planner failed: {e} — using original input")
    return [user_input]

def run_search(queries: List[str], cfg: AgentConfig) -> List[Dict]:
    seen_urls, all_results = set(), []
    for query in queries:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=MAX_RESULTS_PER_QUERY))
            cfg.search_count += 1
            for r in results:
                url = r.get("href", "")
                if url in seen_urls:
                    continue
                seen_urls.add(url)
                all_results.append({
                    "title": r.get("title", "").strip(),
                    "body": re.sub(r'\s+', ' ', r.get("body", ""))[:MAX_BODY_LENGTH],
                    "url": url,
                    "query": query,
                    "word_count": len(r.get("body", "").split())
                })
        except Exception as e:
            log.warning(f"Search failed for '{query}': {e}")
    log.info(f"Search complete — {len(all_results)} unique results")
    return all_results

CREDIBLE_DOMAINS = {
    "wikipedia.org": 10, "bbc.com": 9, "reuters.com": 9,
    "nature.com": 9, "arxiv.org": 8, "techcrunch.com": 7,
    "theverge.com": 7, "wired.com": 7, "github.com": 7,
    "stackoverflow.com": 6, "medium.com": 5
}

def rank_sources(results: List[Dict]) -> List[Dict]:
    def score(r: Dict) -> int:
        url = r.get("url", "").lower()
        domain_score = next((pts for domain, pts in CREDIBLE_DOMAINS.items() if domain in url), 0)
        return domain_score + min(r.get("word_count", 0) // 10, 5)
    ranked = sorted(results, key=score, reverse=True)
    if ranked:
        log.info(f"Top source: {ranked[0]['title'][:60]}")
    return ranked

def summarize_search(user_input: str, results: List[Dict], cfg: AgentConfig) -> str:
    if not results:
        return "No results found."
    results_text = "\n\n".join([
        f"Source {i+1}: {r['title']}\nURL: {r['url']}\nContent: {r['body']}"
        for i, r in enumerate(results)
    ])
    prompt = (
        f"User question: {user_input}\n\nSearch results:\n{results_text}\n\n"
        "Synthesize into a clear, accurate, structured answer. "
        "Mention sources where relevant. Distinguish confirmed facts from uncertain claims."
    )
    try:
        response = client.chat.completions.create(
            model=cfg.model,
            messages=[{"role": "system", "content": cfg.system_prompt},
                      {"role": "user", "content": prompt}],
            max_tokens=cfg.max_tokens,
            temperature=cfg.temperature
        )
        summary = response.choices[0].message.content.strip()
        cfg.session_token_count += response.usage.total_tokens
        log.info(f"Search summary generated")
        return summary
    except Exception as e:
        log.error(f"Summarizer failed: {e}")
        return "Failed to summarize results."

def is_followup(user_input: str, last_query: Optional[str], cfg: AgentConfig) -> Tuple[bool, str]:
    if not last_query:
        return False, ""
    prompt = (
        f"Previous search: {last_query}\nNew question: {user_input}\n"
        "Is this a follow-up? Reply ONLY with JSON: {\"is_followup\": true/false, \"reason\": \"one line\"}"
    )
    try:
        response = client.chat.completions.create(
            model=cfg.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=128,
            temperature=0.1
        )
        parsed = json.loads(response.choices[0].message.content.strip())
        cfg.session_token_count += response.usage.total_tokens
        result = parsed.get("is_followup", False)
        log.info(f"Follow-up detection: {result}")
        return result, last_query if result else ""
    except Exception as e:
        log.warning(f"Follow-up detector failed: {e}")
        return False, ""

log.info("Web search pipeline ready")

2026-06-03 10:38:14,308 [INFO] Web search pipeline ready


In [8]:
# Intent Classifier

INTENT_TYPES = ["chat", "tool", "search", "memory_recall", "file_analysis"]

TOOL_NAMES = ", ".join(TOOL_REGISTRY.keys())

def classify_intent(user_input: str, cfg: AgentConfig) -> Tuple[str, Optional[str], Optional[str]]:
    prompt = (
        "You are an intent classifier for an autonomous AI agent. "
        "Classify the user input into exactly one of these intents: "
        "chat, tool, search, memory_recall, file_analysis.\n\n"
        "Intent definitions:\n"
        "- chat: general conversation, greetings, opinions, explanations\n"
        "- tool: requires one of these tools: " + TOOL_NAMES + "\n"
        "- search: requires live web search for current events or recent information\n"
        "- memory_recall: user asks about something from past conversations\n"
        "- file_analysis: user provides a file path to read or analyze\n\n"
        "If intent is 'tool', also identify the tool name and the input for that tool.\n"
        "Reply ONLY with JSON in this exact format:\n"
        "{\"intent\": \"intent_type\", \"tool_name\": \"tool_name_or_null\", \"tool_input\": \"input_or_null\"}\n\n"
        f"User input: {user_input}"
    )
    try:
        response = client.chat.completions.create(
            model=cfg.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=128,
            temperature=0.1
        )
        raw = response.choices[0].message.content.strip()
        cfg.session_token_count += response.usage.total_tokens
        parsed = json.loads(raw)
        intent = parsed.get("intent", "chat")
        tool_name = parsed.get("tool_name")
        tool_input = parsed.get("tool_input")
        if intent not in INTENT_TYPES:
            intent = "chat"
        cfg.intent_counts[intent] += 1
        log.info(f"Intent classified: {intent} | tool: {tool_name}")
        return intent, tool_name, tool_input
    except Exception as e:
        log.warning(f"Intent classifier failed: {e} — defaulting to chat")
        cfg.intent_counts["chat"] += 1
        return "chat", None, None

log.info("Intent classifier ready")

2026-06-03 10:38:14,332 [INFO] Intent classifier ready


In [9]:
# Orchestrator

_last_search_query: Optional[str] = None

def orchestrate(user_input: str, cfg: AgentConfig) -> Tuple[str, Dict]:
    global _last_search_query
    start_time = time.time()
    metadata = {"intent": None, "tool": None, "memory_hit": False, "sources": []}

    intent, tool_name, tool_input = classify_intent(user_input, cfg)
    metadata["intent"] = intent
    cfg.total_turns += 1

    try:
        if intent == "memory_recall":
            context = memory.retrieve_context(user_input)
            if context:
                cfg.memory_hits += 1
                metadata["memory_hit"] = True
                prompt = f"The user asks: {user_input}\n\nRelevant memory:\n{context}\n\nAnswer using the memory context."
            else:
                prompt = f"The user asks: {user_input}\n\nNo relevant memory found. Answer as best you can."
            response = client.chat.completions.create(
                model=cfg.model,
                messages=[{"role": "system", "content": cfg.system_prompt},
                          {"role": "user", "content": prompt}],
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens

        elif intent == "tool" and tool_name and tool_input:
            tool_result = execute_tool(tool_name, tool_input)
            cfg.tool_call_count += 1
            metadata["tool"] = tool_name
            follow_up_prompt = (
                f"User asked: {user_input}\n"
                f"Tool '{tool_name}' returned: {tool_result}\n"
                "Provide a clear, concise response based on this tool result."
            )
            response = client.chat.completions.create(
                model=cfg.model,
                messages=[{"role": "system", "content": cfg.system_prompt},
                          {"role": "user", "content": follow_up_prompt}],
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens

        elif intent == "search":
            followup, _ = is_followup(user_input, _last_search_query, cfg)
            augmented = f"Previous search: {_last_search_query}\nFollow-up: {user_input}" if followup else user_input
            sub_queries = plan_queries(augmented, cfg)
            raw_results = run_search(sub_queries, cfg)
            ranked = rank_sources(raw_results)
            metadata["sources"] = [r["url"] for r in ranked[:3]]
            reply = summarize_search(user_input, ranked, cfg)
            _last_search_query = user_input

        elif intent == "file_analysis":
            path_match = re.search(r'[A-Za-z]:[/\\][\w/\\. -]+', user_input)
            filepath = path_match.group(0) if path_match else user_input
            metadata["tool"] = "file_reader"
            file_content = file_reader(filepath)
            prompt = (
                f"User request: {user_input}\n"
                f"File content:\n{file_content}\n"
                "Analyze and respond to the user's request about this file."
            )
            response = client.chat.completions.create(
                model=cfg.model,
                messages=[{"role": "system", "content": cfg.system_prompt},
                          {"role": "user", "content": prompt}],
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens

        else:
            context = memory.retrieve_context(user_input)
            if context:
                cfg.memory_hits += 1
                metadata["memory_hit"] = True
            messages = [{"role": "system", "content": cfg.system_prompt + (f"\n\n{context}" if context else "")}]
            messages += memory.short_term.get()
            messages.append({"role": "user", "content": user_input})
            response = client.chat.completions.create(
                model=cfg.model,
                messages=messages,
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens

    except Exception as e:
        log.error(f"Orchestrator pipeline failed: {e} — falling back to chat")
        try:
            response = client.chat.completions.create(
                model=cfg.model,
                messages=[{"role": "system", "content": cfg.system_prompt},
                          {"role": "user", "content": user_input}],
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content.strip()
            cfg.session_token_count += response.usage.total_tokens
            metadata["intent"] = "chat"
        except Exception as e2:
            log.error(f"Fallback also failed: {e2}")
            return "Agent failed to respond.", metadata

    elapsed = round(time.time() - start_time, 2)
    cfg.response_times[metadata["intent"]].append(elapsed)
    memory.add_exchange(user_input, reply)
    log.info(f"Orchestrator complete — intent: {metadata['intent']} | time: {elapsed}s")
    return reply, metadata

log.info("Orchestrator ready")

2026-06-03 10:38:14,368 [INFO] Orchestrator ready


In [10]:
# Response Builder

def build_response(reply: str, metadata: Dict, cfg: AgentConfig) -> str:
    lines = [reply]
    footer_parts = []

    if metadata.get("tool"):
        footer_parts.append(f"tool: {metadata['tool']}")

    if metadata.get("memory_hit"):
        footer_parts.append(f"memory: recalled")

    if metadata.get("sources"):
        sources_str = " | ".join(metadata["sources"][:3])
        footer_parts.append(f"sources: {sources_str}")

    intent = metadata.get("intent", "chat")
    times = cfg.response_times.get(intent, [])
    if times:
        footer_parts.append(f"response time: {times[-1]}s")

    if footer_parts:
        lines.append("\n" + " | ".join(footer_parts))

    return "\n".join(lines)

log.info("Response builder ready")

2026-06-03 10:38:14,386 [INFO] Response builder ready


In [11]:
# Session Logger

class SessionLogger:
    def __init__(self, path: str = SESSION_LOG_PATH):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.entries: List[Dict] = []
        log.info(f"Session logger ready — session id: {self.session_id}")

    def log(self, user_input: str, reply: str, metadata: Dict) -> None:
        self.entries.append({
            "session_id": self.session_id,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "user": user_input,
            "agent": reply,
            "intent": metadata.get("intent"),
            "tool": metadata.get("tool"),
            "memory_hit": metadata.get("memory_hit", False),
            "sources": metadata.get("sources", [])
        })
        try:
            existing = []
            if self.path.exists():
                existing = json.loads(self.path.read_text(encoding="utf-8"))
            existing.extend(self.entries[-1:])
            self.path.write_text(json.dumps(existing, indent=2), encoding="utf-8")
        except Exception as e:
            log.warning(f"Session log write failed: {e}")

    def summary(self) -> str:
        intents = [e["intent"] for e in self.entries]
        counts = {i: intents.count(i) for i in set(intents)}
        return f"Session {self.session_id} | turns: {len(self.entries)} | intents: {counts}"

session_logger = SessionLogger()

2026-06-03 10:38:14,409 [INFO] Session logger ready — session id: 20260603_103814


In [12]:
# Conversation Exporter

class ConversationExporter:
    def __init__(self, path: str = EXPORT_PATH):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def export(self, logger: SessionLogger, cfg: AgentConfig) -> str:
        try:
            lines = [
                "=" * 70,
                "AUTONOMOUS AI AGENT — SESSION EXPORT",
                f"Session ID : {logger.session_id}",
                f"Exported   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
                f"Model      : {cfg.model}",
                f"Total turns: {cfg.total_turns}",
                f"Tokens used: {cfg.session_token_count}",
                f"Tool calls : {cfg.tool_call_count}",
                f"Searches   : {cfg.search_count}",
                f"Memory hits: {cfg.memory_hits}",
                "=" * 70,
                ""
            ]

            for i, entry in enumerate(logger.entries, 1):
                lines += [
                    f"[Turn {i}] {entry['timestamp']}",
                    f"Intent : {entry['intent']}" + (f" | Tool: {entry['tool']}" if entry['tool'] else ""),
                    f"User   : {entry['user']}",
                    f"Agent  : {entry['agent']}",
                    ""
                ]

            avg_times = {
                intent: round(sum(times) / len(times), 2)
                for intent, times in cfg.response_times.items() if times
            }
            lines += [
                "=" * 70,
                "PERFORMANCE SUMMARY",
                f"Avg response times: {avg_times}",
                f"Intent distribution: {cfg.intent_counts}",
                "=" * 70
            ]

            content = "\n".join(lines)
            self.path.write_text(content, encoding="utf-8")
            log.info(f"Session exported to: {self.path}")
            return str(self.path)
        except Exception as e:
            log.error(f"Export failed: {e}")
            return ""

exporter = ConversationExporter()
log.info("Conversation exporter ready")

2026-06-03 10:38:14,434 [INFO] Conversation exporter ready


In [13]:
# Full Agent Master Function

def run_agent(user_input: str, cfg: AgentConfig) -> str:
    reply, metadata = orchestrate(user_input, cfg)
    session_logger.log(user_input, reply, metadata)
    return build_response(reply, metadata, cfg)

def what_can_you_do() -> str:
    lines = ["I am a fully autonomous AI agent. Here is what I can do:\n"]
    for i, cap in enumerate(AGENT_CAPABILITIES, 1):
        lines.append(f"  {i}. {cap}")
    lines += [
        "\nCommands available in this session:",
        "  exit      — end session and export transcript",
        "  memory    — show memory stats",
        "  recall    — semantic search through past conversations",
        "  tools     — list all available tools",
        "  stats     — show session performance stats",
        "  clear     — clear all memory",
        "  help      — show this message"
    ]
    return "\n".join(lines)

log.info("Full agent master function ready")

2026-06-03 10:38:14,450 [INFO] Full agent master function ready


In [14]:
# Interactive Chat Loop

def print_banner():
    print("=" * 70)
    print("  AUTONOMOUS AI AGENT — FULL SYSTEM")
    print(f"  Model    : {config.model}")
    print(f"  Session  : {session_logger.session_id}")
    print(f"  Memory   : {memory.summary()}")
    print(f"  Tools    : {len(TOOL_REGISTRY)} available")
    print("=" * 70)
    print("  Type 'help' to see all capabilities and commands")
    print("=" * 70)
    print()

print_banner()

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() == "exit":
        print("\nExporting session...")
        export_path = exporter.export(session_logger, config)
        print(f"Session exported to: {export_path}")
        print(f"\nSession Summary:")
        print(f"  Turns        : {config.total_turns}")
        print(f"  Tokens used  : {config.session_token_count}")
        print(f"  Tool calls   : {config.tool_call_count}")
        print(f"  Searches     : {config.search_count}")
        print(f"  Memory hits  : {config.memory_hits}")
        print(f"  Intent breakdown: {config.intent_counts}")
        avg_times = {
            intent: round(sum(times) / len(times), 2)
            for intent, times in config.response_times.items() if times
        }
        print(f"  Avg response times: {avg_times}")
        break

    if user_input.lower() == "help":
        print(what_can_you_do())
        continue

    if user_input.lower() == "memory":
        print(f"\n{memory.summary()}\n")
        continue

    if user_input.lower().startswith("recall "):
        query = user_input[7:].strip()
        results = memory.long_term.query(query)
        if results:
            print("\nRecalled memory:")
            for i, r in enumerate(results, 1):
                print(f"\n{i}. {r}")
        else:
            print("\nNo matching memory found.")
        print()
        continue

    if user_input.lower() == "tools":
        print("\nAvailable tools:")
        for name, meta in TOOL_REGISTRY.items():
            print(f"  {name}: {meta['description']}")
        print()
        continue

    if user_input.lower() == "stats":
        avg_times = {
            intent: round(sum(times) / len(times), 2)
            for intent, times in config.response_times.items() if times
        }
        print(f"\nSession stats:")
        print(f"  Turns        : {config.total_turns}")
        print(f"  Tokens       : {config.session_token_count}")
        print(f"  Tool calls   : {config.tool_call_count}")
        print(f"  Searches     : {config.search_count}")
        print(f"  Memory hits  : {config.memory_hits}")
        print(f"  Intents      : {config.intent_counts}")
        print(f"  Avg times    : {avg_times}")
        print(f"  {session_logger.summary()}\n")
        continue

    if user_input.lower() == "clear":
        memory.clear_all()
        print("All memory cleared.\n")
        continue

    reply = run_agent(user_input, config)
    print(f"\nAgent: {reply}\n")

  AUTONOMOUS AI AGENT — FULL SYSTEM
  Model    : llama-3.3-70b-versatile
  Session  : 20260603_103814
  Memory   : Short term: 0/10 messages | Long term: 3 entries | Turns: 0
  Tools    : 10 available
  Type 'help' to see all capabilities and commands



You:  help


I am a fully autonomous AI agent. Here is what I can do:

  1. Multi-turn conversation with persistent memory
  2. Short-term buffer + ChromaDB long-term vector memory
  3. Semantic memory recall across sessions
  4. 10 tools: calculator, wikipedia, web search, file reader, datetime, unit converter, dictionary, weather, csv analyzer, system info
  5. Multi-query web search pipeline with source ranking
  6. Follow-up detection with context injection
  7. Intent classification: chat, tool, search, memory recall, file analysis
  8. Session logging to JSON
  9. Conversation export to .txt
  10. Graceful error recovery with pipeline fallback
  11. Performance tracking per intent type

Commands available in this session:
  exit      — end session and export transcript
  memory    — show memory stats
  recall    — semantic search through past conversations
  tools     — list all available tools
  stats     — show session performance stats
  clear     — clear all memory
  help      — show this

You:  what is my name


2026-06-03 10:39:24,253 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-03 10:39:24,303 [INFO] Intent classified: memory_recall | tool: None
2026-06-03 10:39:25,878 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-03 10:39:26,617 [INFO] Orchestrator complete — intent: memory_recall | time: 2.38s


...

You:  recall Charan



Recalled memory:

1. User: what is my name
Assistant: Your name is Charan.

2. User: what do you know about me
Assistant: I know that your name is Charan and you are currently working on a capstone project. That's the information you've shared with me so far.

3. User: my name is Charan and I am building a capstone project
Assistant: Hello Charan, I've taken note of your introduction. You're currently working on a capstone project. What kind of project is it, and how can I assist you with it?



You:  exit


2026-06-03 10:41:13,660 [INFO] Session exported to: C:\educational files\advanced_agent\memory\session_export.txt



Exporting session...
Session exported to: C:\educational files\advanced_agent\memory\session_export.txt

Session Summary:
  Turns        : 4
  Tokens used  : 3054
  Tool calls   : 2
  Searches     : 3
  Memory hits  : 1
  Intent breakdown: {'chat': 0, 'tool': 2, 'search': 1, 'memory_recall': 1, 'file_analysis': 0}
  Avg response times: {'tool': 1.41, 'search': 13.54, 'memory_recall': 2.38}
